<a href="https://colab.research.google.com/github/comitanigiacomo/Comitani_85673A_Stream_Analysis/blob/main/Comitani_85673A_Stream_anysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Setup Environment

!apt-get update -qq && apt-get install -y openjdk-17-jdk-headless -qq > /dev/null

!test -d spark && echo "Skipping" || (wget -q https://dlcdn.apache.org/spark/spark-4.1.1/spark-4.1.1-bin-hadoop3.tgz -O spark.tgz && mkdir spark && tar xvf spark.tgz -C spark --strip-components=1)
%pip install -q pyspark py4j kaggle mmh3 tqdm

import os

current_dir = os.path.abspath(".")

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = os.path.join(current_dir, "spark")
os.environ['KAGGLE_USERNAME'] = os.getenv('KAGGLE_USERNAME', "xxxxx")
os.environ['KAGGLE_KEY'] = os.getenv('KAGGLE_KEY', "xxxxx")

!test -d nytdataset && echo "Skipping" || kaggle datasets download -d "benjaminawd/new-york-times-articles-comments-2020"
!test -d nytdataset && echo "Skipping" || unzip -q -d nytdataset new-york-times-articles-comments-2020.zip
!rm new-york-times-articles-comments-2020.zip 2> /dev/null

print("Ambiente configurato correttamente")


E: List directory /var/lib/apt/lists/partial is missing. - Acquire (13: Permission denied)
Skipping
Note: you may need to restart the kernel to use updated packages.
Skipping
Skipping
Ambiente configurato correttamente


In [5]:
# Dataset Streaming Interface

# Create a Python generator using yield to simulate a data stream on the dataset.
# I need to implement it like I'm working with a massive dataset, so using a function like pandas.read_csv
# would mean cheating, because all the data would be loaded into RAM.
# My goal is to create a lazy iterator using yield instead of return.

import csv

def stream_all_comments(repo):
    for i in range(10):
        filepath = f"{repo}/nyt-comments-part{i}.csv"
        
        with open(filepath, mode='r', encoding='utf-8') as file:
            reader = csv.reader(file)
            headers = next(reader)
            user_id_index = headers.index("userID")
            for row in reader:
                yield row[user_id_index]

In [ ]:
# Implementation 1

# I want to know in real time how many unique users are interacting today on the site, 
# to understand if there is a bot attack or just to monitor the level of engagement, 
# so I can figure out the best time to publish an article for maximum visibility.
# Obviously, I don't want to keep all the user IDs in RAM.

# Flajolet-Martin Algorithm

# NOTE: For better precision, I will calculate the median of the averages.

import mmh3

class FlajoletMartin:
    def __init__(self, num_hashes=256, num_groups=16):
        self.num_hashes = num_hashes
        self.num_groups = num_groups
        self.output_powers = [1] * self.num_hashes

    def analyze_user_ID(self, item): 
        
        for i in range(self.num_hashes):
            h = mmh3.hash128(str(item), i)
            
            lowest_bit_value = h & -h
            
            if lowest_bit_value > self.output_powers[i]:
                self.output_powers[i] = lowest_bit_value
        
    def get_results(self):
        results = self.output_powers
        
        n = len(results) // self.num_groups
        groups = [results[i:i + n] for i in range(0, len(results), n)]
                                
        means = []
        for group in groups:
            means.append(sum(group) / len(group))
        
        means.sort()
        m = len(means)
        mid = m // 2
        
        if m % 2 != 0:
            raw_estimate = means[mid]
        else:
            raw_estimate = (means[mid-1] + means[mid]) / 2.0
            
        return int(raw_estimate)
    

In [ ]:
repo = "nytdataset/"
FM = FlajoletMartin()

stream = stream_all_comments(repo)


for id_utente in stream:
    FM.analyze_user_ID(id_utente)
    
print("FlajoletMartin Unique Users:", FM.get_results())

FlajoletMartin Unique Users: 1091584


In [ ]:
# Implementation 2: Idea from LogLog version

import mmh3

CORRECTION = 0.77351

class FlajoletMartinLogLog:
    def __init__(self, num_hashes=256, num_groups=16):
        self.num_hashes = num_hashes
        self.num_groups = num_groups
        self.max_zeros = [0] * self.num_hashes

    def analyze_user_ID(self, item): 
        
        for i in range(self.num_hashes):
            h = mmh3.hash128(str(item), i)
            
            # Modification 1: Track the number of trailing zeros instead of the power of two to mitigate variance.
            zeros = (h & -h).bit_length() - 1
            
            if zeros > self.max_zeros[i]:
                self.max_zeros[i] = zeros
        
    def get_results(self):
        results = self.max_zeros
        
        n = len(results) // self.num_groups
        groups = [results[i:i + n] for i in range(0, len(results), n)]
                                
        means = []
        for group in groups:
            means.append(sum(group) / len(group))
        
        means.sort()
        m = len(means)
        mid = m // 2
        
        if m % 2 != 0:
            median_of_means = means[mid]
        else:
            median_of_means = (means[mid-1] + means[mid]) / 2.0
            
        # Modification 2: Apply the power of two to the final median, then scaled by the correction factor to adjust the result.  
        final_estimate = (2 ** median_of_means) * CORRECTION
            
        return int(final_estimate)


In [7]:
repo = "nytdataset/"
FMLL = FlajoletMartinLogLog()

stream = stream_all_comments(repo)


for id_utente in stream:
    FMLL.analyze_user_ID(id_utente)
    
print("FlajoletMartinLogLog Unique Users:", FMLL.get_results())

FlajoletMartinLogLog Unique Users: 423496


In [ ]:
# Bloom Filter

# I want to know if a user is commenting for the first time on an article belonging to a certain section. 
# So I need to check, in the shortest time possible, if the user's ID belongs to the group of users who have already commented in that section.

# Bloom Filter

# NOTE: I need to calculate the optimal dimension of the filter: it depends on the False Positive (FP) rate I want to tolerate.

# I can also implement a test: passing users I know haven't commented, and calculating how many times the filter makes a mistake (false positive), to see if the percentage corresponds to the theoretical formula.

In [ ]:
# Experiments & Scalability Evaluation

# Read around 500,000 lines of data.

# Then I will use a standard Python data structure (like set() to count the actual unique users), 
# so I can make a comparison between the exact count and my algorithm's estimates.

# I can generate a graph to visualize my predictions with respect to the actual values over time.